# Notebook 00 — Environment and First Run

> **阶段**：Stage 1 Use · **预计时间**：20–40 分钟（含模型下载）· **平台**：Kaggle Notebook（开启 GPU）

| 资源 | 估算 | 说明 |
| --- | --- | --- |
| GPU VRAM | 待实测 | 256M 权重约 0.5 GB；P100 会被自动回退 CPU（见 README 排障） |
| GPU 时间 | 实测：模型加载 14 s + 首推理 54 s（CPU） | 含权重下载 |
| 磁盘 | 约 1.5 GB | 模型缓存 |
| Internet | 需要 | 首次下载模型 |


# Learning Objectives

完成本 Notebook 后，你应该能够：

- 在 Kaggle 上检查 GPU 与环境版本并保存快照；
- 用官方推荐接口加载 SmolDocling-256M-preview；
- 完成一次页面图像 → DocTags → DoclingDocument → Markdown/JSON 的完整推理；
- 用自己的话解释 SmolDocling 与传统 OCR 流水线的区别。


# Why This Matters

之后所有 Notebook（数据、基线、训练、评测）都建立在这个最小闭环之上。这里不解决「模型好不好」，而是保证「从图像到结构化输出的链路可复现」。版本快照是本课程的实验规范：没有环境记录的实验结果不可信。


# Concepts

```text
Document Image
      ↓
Vision Encoder
      ↓
Multimodal Model（Idefics3 架构）
      ↓
Structured Generation
      ↓
DocTags（结构化中间表示）
      ↓
DoclingDocument → Markdown / JSON / HTML
```

- **DocTags**：SmolDocling 的输出格式，同时携带文本、结构、坐标与阅读顺序（延伸阅读：`docs/reading/04_doctags.md`）；
- **DoclingDocument**：Docling 生态的统一文档对象，DocTags 是其一种输入/导出表示；
- **官方 Prompt**：`Convert this page to docling.`（模型卡原样）。


## Step 1 — 检查 Kaggle GPU 与环境

先运行下面的环境快照。它自动记录 GPU 型号、VRAM、CUDA、PyTorch、transformers、docling-core 版本——这些信息会进入之后每次实验的元数据。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src.inference import environment_snapshot
import json
print(json.dumps(environment_snapshot(), ensure_ascii=False, indent=2))


## Step 2 — 安装依赖

依赖清单固定在本仓库 `requirements-kaggle.txt`。Kaggle 镜像自带 torch/transformers 时不要重装；缺哪个装哪个。首次运行后，把实际生效的版本回填到清单注释里（见 `docs/notebook-design.md` §7 的策略）。


In [ ]:
# 首次运行时取消注释（逐条安装，避免覆盖镜像自带版本）：
# !pip install docling-core huggingface_hub pyyaml --quiet
# !pip show transformers docling-core | grep -E "^(Name|Version)"


## Step 3 — 加载 SmolDocling-256M-preview


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src.model import SmolDoclingAdapter, model_summary

adapter = SmolDoclingAdapter().load()  # 默认 device=auto, dtype=auto, attention=auto
print(model_summary(adapter))


## Step 4 — 准备一张简单文档图像

为了不依赖外部文件，我们用 PIL 在本地合成一页「标题 + 段落 + 表格 + 公式」的简单文档。真实页面在 Notebook 01 用 OmniDocBench 数据替换。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from PIL import Image, ImageDraw, ImageFont

def make_sample_document():
    img = Image.new('RGB', (900, 1000), 'white')
    d = ImageDraw.Draw(img)
    try:
        font_title = ImageFont.truetype('DejaVuSans-Bold.ttf', 42)
        font_body = ImageFont.truetype('DejaVuSans.ttf', 24)
    except OSError:
        font_title = font_body = ImageFont.load_default()
    d.text((60, 50), 'SmolDocling Test Document', fill='black', font=font_title)
    d.text((60, 130), 'This page is a simple synthetic sample generated locally.', fill='black', font=font_body)
    d.text((60, 180), 'It contains a title, a paragraph and a small table.', fill='black', font=font_body)
    rows = [('Item', 'Quantity', 'Price'), ('Pen', '2', '1.50'), ('Paper', '1', '4.00'), ('Total', '', '5.50')]
    for r, row in enumerate(rows):
        for c, cell in enumerate(row):
            x, y = 60 + c * 260, 260 + r * 48
            d.rectangle([x, y, x + 250, y + 40], outline='black', width=2)
            d.text((x + 8, y + 8), cell, fill='black', font=font_body)
    d.text((60, 520), 'E = mc^2', fill='black', font=font_body)
    return img

sample_image = make_sample_document()
display(sample_image)


## Step 5 — 第一次推理


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

prediction = adapter.predict(sample_image)
print('latency_sec =', prediction['latency_sec'])
print('device =', prediction['device'], '| dtype =', prediction['dtype'])


## Step 6 — 看四种输出：raw / DocTags / Markdown / JSON

SmolDocling 直接生成的是 **DocTags**（带结构、坐标、阅读顺序的标记文本），再经 `docling-core` 转成 `DoclingDocument`，最后导出 Markdown / JSON / HTML。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

# 6.1 raw generation（模型原始输出）
print(prediction['doctags'][:2000])


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

# 6.2 DocTags -> DoclingDocument -> Markdown / JSON
from src.model import doctags_to_docling

doc, api_path = doctags_to_docling(prediction['doctags'], sample_image)
print('实际生效的转换 API:', api_path)  # docling-core 新旧 API 兼容（设计文档风险 R2）

markdown = doc.export_to_markdown()
print('----- Markdown -----')
print(markdown[:1500])

doc_dict = doc.export_to_dict()
print('----- DoclingDocument 顶层字段 -----')
print(list(doc_dict.keys()))


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import json
out_dir = REPO_ROOT / 'results' / 'nb00'
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'first_run.md').write_text(markdown, encoding='utf-8')
(out_dir / 'first_run_doctags.txt').write_text(prediction['doctags'], encoding='utf-8')
(out_dir / 'first_run_doc.json').write_text(json.dumps(doc_dict, ensure_ascii=False, indent=2), encoding='utf-8')
(out_dir / 'metadata.json').write_text(json.dumps(prediction, ensure_ascii=False, indent=2), encoding='utf-8')
print('已保存:', out_dir)


## Step 7 — SmolDocling 到底在做什么

SmolDocling 是一个 **端到端 VLM**：它不把 OCR、版面分析、表格识别拆成独立模块，而是直接学习「页面图像 + 指令 → DocTags」。因此：

- 传统 OCR 输出纯文本，版面、表格、公式、阅读顺序需要额外模块恢复，误差逐级累积；
- SmolDocling 输出结构化 DocTags，文本与结构在同一生成过程中联合建模；
- 代价是输出可能包含幻觉，结构稳定性需要下游验证（这正是后面 Benchmark / Error Analysis 要研究的问题）。

| | 传统 Modular OCR | SmolDocling（端到端 VLM） |
| --- | --- | --- |
| 输出 | 文本行/框 | DocTags（文本+结构+坐标+顺序） |
| 结构恢复 | 后置模块 | 联合生成 |
| 可检查性 | 每步可查 | 需解析/验证输出 |
| 幻觉风险 | 低 | 需要专门评估 |


# What You Should Observe

- DocTags 里能同时看到 `<loc_*>` 坐标、`<table>`/`<otsl>`、`<formula>` 等结构标记；
- 同一份 DocTags 可以无损导出 Markdown 与 JSON，二者信息量不同（JSON 保留更多结构）；
- `metadata.json` 记录了 model revision、prompt、latency——这就是实验证据链的起点。


# Research Checkpoint

回答下面的问题（教师参考答案见 `solutions/nb00_answers.md`，先自己想再对照）：

> **SmolDocling 与传统 OCR 最大的区别是什么？** 从输出内容、误差传播方式和需要验证的风险三个方面回答。

**TODO：** 把你的答案写在 `results/nb00/research_checkpoint.md`。


# Exercises

1. **TODO：** 修改合成文档：加一个列表、把表格换成三线表样式，重跑推理，观察 DocTags 对应部分有什么变化；
2. **TODO：** 把 `model_summary(adapter)` 与 `environment_snapshot()` 的结果截图/记录到学习日志，标注实际 GPU 型号与 VRAM；
3. **TODO：** 用 `adapter.predict(sample_image, prompt='Convert this page to docling.')` 与 `max_new_tokens=1024` 各跑一次，记录 latency 与输出截断现象，说明原因。


# Takeaways

- 可复现实验从环境快照开始；
- SmolDocling 的输出链路是 图像 → DocTags → DoclingDocument → 多格式导出；
- 端到端 VLM 与传统 OCR 的差别决定了后面必须单独评测幻觉与结构错误。

**下一步**：[Notebook 01](01_Understanding_OmniDocBench.ipynb) — 理解我们将要评测的数据。
